# 04 — Final Model and Bundle

## 1. Finalization Context and Boundary

Notebook 03 froze model selection. This thin presentation/orchestration layer authenticates its handoff, delegates finalization to the production runner, and presents persisted aggregate evidence.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        handoff_path = candidate / 'artifacts/model-selection/concrete-compressive-strength/model-selection-handoff.json'
        if (candidate / 'scripts/finalize_model.py').is_file() and handoff_path.is_file():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

PROJECT_ROOT = find_project_root(Path.cwd())
SELECTION_HANDOFF = Path('artifacts/model-selection/concrete-compressive-strength/model-selection-handoff.json')
OUTPUT_DIR = Path('artifacts/models/concrete-compressive-strength')
from scripts.select_models import load_and_validate_model_selection_handoff
from scripts.finalize_model import (
    load_and_validate_final_model_handoff, load_and_validate_final_model_manifest,
    load_and_validate_final_test_evidence, load_and_validate_inference_bundle,
    load_trusted_pipeline_from_bundle, run_regression_finalization,
)
selection_handoff = load_and_validate_model_selection_handoff(project_root=PROJECT_ROOT, handoff_path=SELECTION_HANDOFF)

## 2. Frozen Model-Selection Contract

The authenticated model-selection-handoff.v3 is the independent handoff from Notebook 03. Every value below comes from that frozen contract.

In [2]:
target_contract = selection_handoff['target_contract']
cv_contract = selection_handoff['cv_contract']
training_instructions = selection_handoff['final_training_instructions']
contract_rows = [
 ('Selected model ID', selection_handoff['selected_model_id']),
 ('Model family', selection_handoff['selected_model_family']),
 ('Feature policy', selection_handoff['selected_feature_policy']),
 ('Feature count', len(selection_handoff['selected_feature_columns'])),
 ('Target', target_contract['column']), ('Target semantics', target_contract['semantics']),
 ('Target unit', target_contract['unit']), ('Primary metric', selection_handoff['primary_metric'].upper()),
 ('Metric direction', selection_handoff['primary_metric_direction']),
 ('CV strategy', f"{cv_contract['strategy']} ({cv_contract['n_splits']} folds; fit={cv_contract['fit_partition']})"),
 ('Future training partitions', ', '.join(training_instructions['fit_partitions'])),
 ('Evaluation partition', training_instructions['final_evaluation_partition']),
 ('Test sealing state', 'sealed' if selection_handoff['test_partition_sealed'] else 'not sealed'),
]
display(pd.DataFrame(contract_rows, columns=['Contract field', 'Frozen value']))
display(pd.DataFrame(selection_handoff['selected_hyperparameters'].items(), columns=['Selected hyperparameter', 'Value']))
display(pd.DataFrame(selection_handoff['selected_estimator_fixed_constructor_parameters'].items(), columns=['Fixed constructor parameter', 'Value']))
display(pd.DataFrame({'Position': range(1, len(selection_handoff['selected_feature_columns']) + 1), 'Ordered feature': selection_handoff['selected_feature_columns']}))

,Contract field,Frozen value
0,Selected model ID,hist_gradient_boosting
1,Model family,HistGradientBoostingRegressor
2,Feature policy,all_features
3,Feature count,8
4,Target,Concrete compressive strength
5,Target semantics,Continuous / quantitative
6,Target unit,MPa
7,Primary metric,MAE
8,Metric direction,lower_is_better
9,CV strategy,KFold (5 folds; fit=train_only)


,Selected hyperparameter,Value
0,model__l2_regularization,1.0
1,model__learning_rate,0.1
2,model__max_leaf_nodes,15.0
3,model__min_samples_leaf,10.0


,Fixed constructor parameter,Value
0,categorical_features,from_dtype
1,early_stopping,auto
2,l2_regularization,0.0
3,learning_rate,0.1
4,loss,squared_error
5,max_bins,255
6,max_features,1.0
7,max_iter,300
8,max_leaf_nodes,31
9,min_samples_leaf,20


,Position,Ordered feature
0,1,Cement
1,2,Blast Furnace Slag
2,3,Fly Ash
3,4,Water
4,5,Superplasticizer
5,6,Coarse Aggregate
6,7,Fine Aggregate
7,8,Age


## 3. Final Training and One-Time Test Policy

    Train + Validation
            ↓
    single final fit
            ↓
    verified fitted model freeze
            ↓
    first allowed test access
            ↓
    one test prediction
            ↓
    aggregate final evidence

Notebook 04 does not retune. Test is final evaluation, not selection; its results cannot alter the frozen model.

## 4. Atomic Finalization

All operational work remains centralized in run_regression_finalization(...). Complete equivalent artifacts are reused; incomplete or divergent state fails closed.

In [3]:
finalization_result = run_regression_finalization(
    project_root=PROJECT_ROOT, model_selection_handoff_path=SELECTION_HANDOFF, output_directory=OUTPUT_DIR,
)
manifest = load_and_validate_final_model_manifest(project_root=PROJECT_ROOT, manifest_path=OUTPUT_DIR / 'final-model-manifest.json')
test_evidence = load_and_validate_final_test_evidence(project_root=PROJECT_ROOT, evidence_path=OUTPUT_DIR / 'final-test-evidence.json')
bundle = load_and_validate_inference_bundle(project_root=PROJECT_ROOT, bundle_path=OUTPUT_DIR / 'inference-bundle.json')
final_handoff = load_and_validate_final_model_handoff(project_root=PROJECT_ROOT, handoff_path=OUTPUT_DIR / 'final-model-handoff.json')
evidence_rows = [
 ('Artifact set status', finalization_result.get('status')),
 ('Final fit count in this call', finalization_result.get('final_fit_count')),
 ('Test parse/access count in this call', finalization_result.get('test_parse_count')),
 ('Test prediction count in this call', finalization_result.get('test_predict_count')),
 ('Persisted test evaluation count', finalization_result.get('test_evaluation_count', manifest.get('test_evaluation_count'))),
 ('Serialization roundtrip verified', finalization_result.get('roundtrip_verified')),
 ('Model state fingerprint', finalization_result.get('model_state_fingerprint', bundle.get('model_state_fingerprint'))),
 ('No post-test adjustment', test_evidence.get('no_post_test_adjustment')),
]
display(pd.DataFrame([(k, v) for k, v in evidence_rows if v is not None], columns=['Execution evidence', 'Runner / persisted value']))
counts = manifest['frozen_finalization_contract']['training_row_counts']
display(pd.DataFrame([('Train', counts['train']), ('Validation', counts['validation']), ('Final training (train + validation)', manifest['training_row_count']), ('Test', test_evidence['row_count'])], columns=['Partition', 'Rows']))

,Execution evidence,Runner / persisted value
0,Artifact set status,reused_equivalent
1,Final fit count in this call,0
2,Test parse/access count in this call,0
3,Test prediction count in this call,0
4,Persisted test evaluation count,1
5,Model state fingerprint,9c14201607cdfcf0e92ac902361fa6815641e3c143c5a4...
6,No post-test adjustment,True


,Partition,Rows
0,Train,721
1,Validation,154
2,Final training (train + validation),875
3,Test,155


## 5. Final Test Performance

Frozen validation is the evidence used in selection; final test is the independent evaluation. Persisted deltas are descriptive only and never cause retuning or reselection. No row-level test data is loaded.

In [4]:
validation_metrics = selection_handoff['selected_validation_evidence']
final_metrics = test_evidence['metrics']
persisted_deltas = test_evidence['validation_to_test_deltas']
metric_specs = [('MAE','mae','test_mae_minus_validation_mae'), ('RMSE','rmse','test_rmse_minus_validation_rmse'), ('R²','r2','test_r2_minus_validation_r2'), ('MedAE','medae','test_medae_minus_validation_medae')]
performance = pd.DataFrame([(label, validation_metrics[key], final_metrics[key], persisted_deltas[delta]) for label,key,delta in metric_specs], columns=['Metric','Frozen validation','Final test','Test − Validation'])
display(performance.style.format({column: '{:.4f}' for column in performance.columns[1:]}))
diagnostic_specs = [('Residual mean','residual_mean'), ('Residual standard deviation','residual_standard_deviation'), ('Maximum absolute error','max_absolute_error'), ('Absolute error p50','absolute_error_p50'), ('Absolute error p90','absolute_error_p90'), ('Absolute error p95','absolute_error_p95')]
diagnostics = pd.DataFrame([(label, final_metrics[key]) for label,key in diagnostic_specs], columns=['Aggregate error diagnostic','Final test value'])
display(diagnostics.style.format({'Final test value': '{:.4f}'}))

,Metric,Frozen validation,Final test,Test − Validation
0,MAE,2.7417,2.5822,-0.1595
1,RMSE,4.0870,4.2104,0.1234
2,R²,0.9336,0.9387,0.0050
3,MedAE,1.8033,1.6363,-0.1669


,Aggregate error diagnostic,Final test value
0,Residual mean,0.7577
1,Residual standard deviation,4.1551
2,Maximum absolute error,26.6252
3,Absolute error p50,1.6363
4,Absolute error p90,6.4306
5,Absolute error p95,7.7465


## 6. Final Model and Inference Artifacts

Public loaders validate the JSON contracts. The bundle authenticates the model reference; Section 8 verifies its SHA-256 before trusted loading. Runner status applies to the set, not each artifact.

In [5]:
import hashlib

def abbreviated_sha(value):
    return f'{value[:16]}…' if value else 'not embedded'

handoff_sha = hashlib.sha256((PROJECT_ROOT / OUTPUT_DIR / 'final-model-handoff.json').read_bytes()).hexdigest()
artifact_rows = [
 ('final-pipeline.joblib','joblib model artifact',bundle['model_artifact_path'],bundle['model_artifact_sha256'],'validated reference'),
 ('final-model-manifest.json',manifest['schema_version'],final_handoff['manifest_reference']['path'],final_handoff['manifest_reference']['sha256'],'validated'),
 ('final-test-evidence.json',test_evidence['schema_version'],final_handoff['test_evidence_reference']['path'],final_handoff['test_evidence_reference']['sha256'],'validated'),
 ('inference-bundle.json',bundle['schema_version'],final_handoff['bundle_reference']['path'],final_handoff['bundle_reference']['sha256'],'validated'),
 ('final-model-handoff.json',final_handoff['schema_version'],str(OUTPUT_DIR / 'final-model-handoff.json'),handoff_sha,'validated'),
]
artifact_table = pd.DataFrame(artifact_rows, columns=['Artifact','Schema / Format','Path','SHA-256','Validation state'])
artifact_table['SHA-256'] = artifact_table['SHA-256'].map(abbreviated_sha)
display(artifact_table)
display(pd.DataFrame([('Artifact set status', finalization_result.get('status'))], columns=['Set-level state','Value']))

,Artifact,Schema / Format,Path,SHA-256,Validation state
0,final-pipeline.joblib,joblib model artifact,artifacts/models/concrete-compressive-strength...,6e6a5a970c6e91b4…,validated reference
1,final-model-manifest.json,final-model-manifest.v3,artifacts/models/concrete-compressive-strength...,e42c52d91334ec90…,validated
2,final-test-evidence.json,final-test-evidence.v3,artifacts/models/concrete-compressive-strength...,71185852c04cb00b…,validated
3,inference-bundle.json,inference-bundle.v3,artifacts/models/concrete-compressive-strength...,5445a88ff4ba5879…,validated
4,final-model-handoff.json,final-model-handoff.v3,artifacts/models/concrete-compressive-strength...,13fb495eace8cd84…,validated


,Set-level state,Value
0,Artifact set status,reused_equivalent


## 7. Continuous Regression Inference Contract

The bundle is the machine-readable consumer contract passed from Notebook 04 to Notebook 05.

In [6]:
prediction_contract = bundle['prediction_contract']
bundle_target = bundle['target_contract']
preprocessing = bundle['preprocessing_contract']
inference_rows = [
 ('Prediction type', prediction_contract['type']), ('Target', bundle_target['column']),
 ('Target semantics', bundle_target['semantics']), ('Unit', prediction_contract['unit']),
 ('Input feature count', len(bundle['feature_order'])), ('Ordered features', ' → '.join(bundle['feature_order'])),
 ('Output scale', prediction_contract['scale']), ('Model family', bundle['selected_model_family']),
 ('Preprocessing', f"{preprocessing['type']}; scale numerical={preprocessing['scale_numerical']}; fit scope={preprocessing['fit_scope']}"),
]
display(pd.DataFrame(inference_rows, columns=['Inference field','Validated value']))

,Inference field,Validated value
0,Prediction type,continuous_numeric
1,Target,Concrete compressive strength
2,Target semantics,Continuous / quantitative
3,Unit,MPa
4,Input feature count,8
5,Ordered features,Cement → Blast Furnace Slag → Fly Ash → Water ...
6,Output scale,original_target_scale
7,Model family,HistGradientBoostingRegressor
8,Preprocessing,Pipeline; scale numerical=False; fit scope=tra...


Classification concepts do not apply to continuous regression: there are no classes, positive-class semantics, classification threshold, or argmax class decision. The output is continuous numeric.

## 8. Trusted Artifact Reload

This check occurs in the current notebook kernel. The loader verifies model SHA-256 before deserialization, validates fitted state, and makes no prediction.

In [7]:
reload_handoff = load_and_validate_final_model_handoff(project_root=PROJECT_ROOT, handoff_path=OUTPUT_DIR / 'final-model-handoff.json')
reload_bundle = load_and_validate_inference_bundle(project_root=PROJECT_ROOT, bundle_path=OUTPUT_DIR / 'inference-bundle.json')
trusted_pipeline = load_trusted_pipeline_from_bundle(project_root=PROJECT_ROOT, bundle=reload_bundle)
reload_rows = [('Final handoff schema',reload_handoff['schema_version']), ('Inference bundle schema',reload_bundle['schema_version']), ('Model class',trusted_pipeline.__class__.__name__), ('Model SHA verified before trusted load',True), ('Prediction contract',reload_bundle['prediction_contract'])]
display(pd.DataFrame(reload_rows, columns=['Trusted reload evidence','Value']))

,Trusted reload evidence,Value
0,Final handoff schema,final-model-handoff.v3
1,Inference bundle schema,inference-bundle.v3
2,Model class,Pipeline
3,Model SHA verified before trusted load,True
4,Prediction contract,"{'scale': 'original_target_scale', 'type': 'co..."


## 9. Final Handoff and Readiness

Only fields actually persisted in final_model_handoff readiness are shown.

In [8]:
readiness = final_handoff['readiness']
readiness_fields = ['model_selection_handoff_validated','selected_candidate_reconstructed','frozen_finalization_contract_validated','final_training_completed','final_model_trained','final_fit_count','test_partition_opened_after_final_fit','final_test_evaluation_completed','test_partition_evaluated','test_partition_evaluation_count','test_prediction_call_count','final_model_artifact_materialized','inference_bundle_materialized','inference_demo_ready','operational_modeling_ready','operational_validity']
display(pd.DataFrame([(field, readiness[field]) for field in readiness_fields if field in readiness], columns=['Readiness field','Persisted value']))

,Readiness field,Persisted value
0,model_selection_handoff_validated,True
1,selected_candidate_reconstructed,True
2,frozen_finalization_contract_validated,True
3,final_training_completed,True
4,final_model_trained,True
5,final_fit_count,1
6,test_partition_opened_after_final_fit,True
7,final_test_evaluation_completed,True
8,test_partition_evaluated,True
9,test_partition_evaluation_count,1


The educational study has reached its final model and bundle. Test has already been used exactly as final independent evaluation, and Notebook 05 can act as an independent consumer. Persisted operational_modeling_ready=false and operational_validity=unconfirmed remain unchanged: this is **not** production readiness.